# SQL Feature Engineering

This notebook will be concerned with forming our Djokovic-specific dataset. We will extract his matches, wrangle existing features in the dataset into Djokovic-based features that a model can easily interpret (such as turning the winner column into a binary column indicating whether Djokovic won or not), and engineering new features. We use DuckDB for SQL usage.

In [1]:
import pandas as pd
import duckdb as db

## 1. Extracting Djokovic Matches

We begin with obtaining Djokovic's matches. We initiate our connection with a DuckDB database, read our overall ATP tennis data csv, and create a new table with only Djokovic's matches.

In [2]:
con = db.connect("tennis_db.db")

In [3]:
con.sql(""" CREATE OR REPLACE TABLE raw_matches AS
    SELECT * FROM read_csv_auto('../Data/atp_tennis.csv')
""")

In [4]:
con.sql(""" SELECT * FROM RAW_MATCHES""").df().head()

,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Arthurs W.,Gambill J.M.,Gambill J.M.,105,58,-1,-1,-1.0,-1.0,6-3 6-7 4-6
1,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Balcells J.,Henman T.,Henman T.,218,11,-1,-1,-1.0,-1.0,4-6 6-7
2,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Clement A.,Enqvist T.,Enqvist T.,56,5,-1,-1,-1.0,-1.0,3-6 3-6
3,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Dosedel S.,Ljubicic I.,Dosedel S.,63,77,-1,-1,-1.0,-1.0,6-4 6-2
4,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Draper S.,Norman M.,Norman M.,155,15,-1,-1,-1.0,-1.0,4-6 4-6


In [5]:
con.sql(""" CREATE OR REPLACE TABLE djok_matches AS
SELECT * FROM RAW_MATCHES 
WHERE Player_1 = 'Djokovic N.'
OR Player_2 = 'Djokovic N.' """)

In [6]:
con.sql(""" SELECT * FROM djok_matches """).df().head()

,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,Croatia Open,2004-07-20,International,Outdoor,Clay,1st Round,3,Djokovic N.,Volandri F.,Volandri F.,368,65,-1,-1,5.50,1.12,6-7 1-6
1,Open Romania,2004-09-14,International,Outdoor,Clay,1st Round,3,Clement A.,Djokovic N.,Djokovic N.,67,272,-1,-1,1.53,2.35,6-2 4-6 4-6
2,Open Romania,2004-09-16,International,Outdoor,Clay,2nd Round,3,Ferrer D.,Djokovic N.,Ferrer D.,54,272,-1,-1,1.28,3.50,4-6 6-4 6-4
3,Thailand Open,2004-09-28,International,Indoor,Hard,1st Round,3,van Scheppingen D.,Djokovic N.,van Scheppingen D.,81,248,-1,-1,1.61,2.20,4-6 7-5 6-4
4,Australian Open,2005-01-17,Grand Slam,Outdoor,Hard,1st Round,5,Djokovic N.,Safin M.,Safin M.,188,4,-1,-1,7.50,1.07,0-6 2-6 1-6


In [7]:
df_djok = con.sql(""" FROM djok_matches """).df()
verdicts = (df_djok["Player_1"] == 'Djokovic N.') | (df_djok["Player_2"] == 'Djokovic N.')
print(f"All matches involve Djokovic: {verdicts.all()}")

All matches involve Djokovic: True


The above check shows that all of the matches we have extracted involve Djokovic. We should also check that we haven't left out any of Djokovic's matches. We will do this by comparing the number of matches with Djokovic in them in the overall dataset with the size of our extracted dataset. Combined with the fact that we know that all of our extracted matches are Djokovic's, this means we haven't left any out.

In [8]:
con.sql(""" SELECT COUNT(*) FROM raw_matches
            WHERE Player_1 = 'Djokovic N.' OR Player_2 = 'Djokovic N.' """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1278 │
└──────────────┘

So there are $1278$ matches with Djokovic playing in the original dataset.

In [9]:
print(f"Number of Djokovic matches extracted: {df_djok.shape[0]}")

Number of Djokovic matches extracted: 1278


And the above shows we have extracted all of those matches.

## 2. Existing Feature Wrangling

Before moving on to developing new features, we want to turn the features already in the dataset into Djokovic-specific features. Features such as odds, rank and points all have two such columns corresponding to columns Player_1 and Player_2. Since the original dataset arbitrarily chose Djokovic as player 1 or 2, his data can be either odd_1 or odd_2, and the same for rank and pts. We need to turn these arbitrary features into Djokovic-specific odds, ranks and points columns so the models can have a clear understanding of Djokovic's features and the opponent's features, rather than a confusing situation where sometimes Djokovic's points, odds and rank are in one column, and other times in another. 

In [10]:
con.sql(""" CREATE OR REPLACE TABLE djok_matches AS
SELECT *,

CASE WHEN Winner = 'Djokovic N.' THEN 1
ELSE 0 END AS dj_win,

CASE WHEN Player_1 = 'Djokovic N.' THEN Rank_1
ELSE Rank_2 END AS dj_rank,

CASE WHEN Player_1 = 'Djokovic N.' THEN Rank_2
ELSE Rank_1 END AS opp_rank,

CASE WHEN Player_1 = 'Djokovic N.' THEN Pts_1
ELSE Pts_2 END AS dj_pts,

CASE WHEN Player_1 = 'Djokovic N.' THEN Pts_2
ELSE Pts_1 END AS opp_pts,

CASE WHEN Player_1 = 'Djokovic N.' THEN Odd_1
ELSE Odd_2 END AS dj_odds,

CASE WHEN Player_1 = 'Djokovic N.' THEN Odd_2
ELSE Odd_1 END AS opp_odds,

CASE WHEN Court = 'Outdoor' THEN 1
ELSE 0 END AS is_outdoor,

CASE WHEN "Best of" = 5 THEN 1
ELSE 0 END AS is_five_sets

FROM djok_matches """)

In [11]:
con.sql(""" SELECT * FROM djok_matches """).df().head()

,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,...,Score,dj_win,dj_rank,opp_rank,dj_pts,opp_pts,dj_odds,opp_odds,is_outdoor,is_five_sets
0,Croatia Open,2004-07-20,International,Outdoor,Clay,1st Round,3,Djokovic N.,Volandri F.,Volandri F.,...,6-7 1-6,0,368,65,-1,-1,5.50,1.12,1,0
1,Open Romania,2004-09-14,International,Outdoor,Clay,1st Round,3,Clement A.,Djokovic N.,Djokovic N.,...,6-2 4-6 4-6,1,272,67,-1,-1,2.35,1.53,1,0
2,Open Romania,2004-09-16,International,Outdoor,Clay,2nd Round,3,Ferrer D.,Djokovic N.,Ferrer D.,...,4-6 6-4 6-4,0,272,54,-1,-1,3.50,1.28,1,0
3,Thailand Open,2004-09-28,International,Indoor,Hard,1st Round,3,van Scheppingen D.,Djokovic N.,van Scheppingen D.,...,4-6 7-5 6-4,0,248,81,-1,-1,2.20,1.61,0,0
4,Australian Open,2005-01-17,Grand Slam,Outdoor,Hard,1st Round,5,Djokovic N.,Safin M.,Safin M.,...,0-6 2-6 1-6,0,188,4,-1,-1,7.50,1.07,1,1


For now, we will keep the original features in the data, but we will delete the now redundant columns once we have completed all of our feature engineering.

In $2009$, ATP changed the names of some of the series titles. Specifically, 'International' became 'ATP250', 'International Gold' became 'ATP500', and 'Masters' became 'Masters 1000'. Since these still describe the same tournaments, and to prevent a model thinking that these describe distinct series types, we will rename the legacy series names to the current ones.

In [12]:
con.sql(""" UPDATE djok_matches
SET Series = CASE WHEN Series = 'International' THEN 'ATP250'
                  WHEN Series = 'International Gold' THEN 'ATP500'
                  WHEN Series = 'Masters' THEN 'Masters 1000'
                  ELSE Series 
                  END """)

In [13]:
con.sql(""" SELECT DISTINCT Series FROM djok_matches """)

┌──────────────┐
│    Series    │
│   varchar    │
├──────────────┤
│ ATP500       │
│ ATP250       │
│ Masters 1000 │
│ Grand Slam   │
│ Masters Cup  │
└──────────────┘

We can see above that we have successfully turned all of the legacy series names into the new ones, since the old names no longer appear in the distinct series name call.

## 3. New Features

Having turned the existing features into formats that a model can learn, we now move on to creating new features. We begin with turning the date column into a numeric age column. We also create a quadratic age column since the relationship between age and the performance of an athlete is often approximately negatively quadratic (rising performance at the start of career, peak in the middle, then drop off towards the end).

In [14]:
con.sql(""" CREATE OR REPLACE TABLE djok_matches AS
SELECT *,
DATEDIFF('day', '1987/05/22', Date) / 365.25 AS age,
age - AVG(age) OVER() AS age_c,
age_c ** 2 AS age_c_sq
FROM djok_matches
"""
)

Next, we want to start creating further features. One of these features will be a player-specific head to head win rate. In order to make this easier, we first create a column that names Djokovic's opponent. If we didn't do this, then the opponent would be in either of the player columns, and grouping by either of these columns to extract the average rate would be much more difficult.

In [15]:
con.sql(""" CREATE OR REPLACE TABLE djok_matches AS
SELECT *,
CASE WHEN Player_1 = 'Djokovic N.' THEN Player_2
ELSE Player_1 END AS opp_name
FROM djok_matches""")

Now for the main part of engineering our features, which will be a five-match simple moving average, surface-specific form over the last twenty matches, a head to head win rate for all matches against the specific opponent, and the number of matches he has played in the last $30$ days.
 
We use a five-match moving average to give an idea of Djokovic's current short-term form. The surface-specific form will provide models with his current performance on the surface of the next match. The head to head win rate provides information on how the next specific player matchup has gone for Djokovic historically. The number of matches played in the last 30 days will act as a proxy for Djokovic's fatigue in the next match.

For the head to head win rate, we often have situations where Djokovic has never played the next opponent before. In such cases, we use Djokovic's overall historic win rate.

In [16]:
con.sql(""" CREATE OR REPLACE TABLE djok_matches AS
SELECT *,

AVG(dj_win) OVER(ORDER BY Date 
                    ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS "five_sma",

AVG(dj_win) OVER(PARTITION BY Surface 
                    ORDER BY Date 
                    ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS "surface_form_last_20",

COALESCE(AVG(dj_win) OVER(PARTITION BY opp_name 
                    ORDER BY Date
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),
                    AVG(dj_win) OVER(ORDER BY Date 
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS "head_to_head_rate",

COUNT(*) OVER(ORDER BY Date
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING) AS "matches_last_30"

FROM djok_matches""")

## 4. Dropping Redundant Columns and Exporting CSV

Having now engineered our Djokovic-specific columns and new columns, we can drop the ones we no longer need. We also drop the Tournament column since it is high-cardinality meaning one-hot encoding would be ineffective, and much of the information it contains is redundant with age, series and surface.

In [17]:
con.sql("""
    CREATE OR REPLACE TABLE djok_matches AS
    SELECT * EXCLUDE (
        Tournament, Date, Court, "Best of", Player_1, Player_2, Winner, Score, age,
        Rank_1, Rank_2, Pts_1, Pts_2, Odd_1, Odd_2, opp_name
    )
    FROM djok_matches
""")


Next, we also notice that some matches historically were played on carpet, however there are very little of these compared to the amount of other surfaces, and they all happen at earlier dates. If we keep these, when we later do a chronological train/test split, we will be training on some surfaces that never appear in the test set. This is unnecessary, and we will drop these rows.

In [18]:
con.sql(""" DELETE FROM djok_matches
            WHERE Surface = 'Carpet' """)

In [19]:
con.sql(""" SELECT DISTINCT Surface FROM djok_matches """)

┌─────────┐
│ Surface │
│ varchar │
├─────────┤
│ Clay    │
│ Hard    │
│ Grass   │
└─────────┘

The above shows we no longer have the carpet matches in the table. Note that we couldn't do this carpet deletion before calculating the rolling features such as the five-match average since these matches did still happen and will contribute to such features.

We have finished our data manipulation, and we can now export it.

In [20]:
con.sql("""
    COPY (SELECT * FROM djok_matches ORDER BY age_c) TO '../Data/djok_data.csv' (HEADER, DELIMITER ',')
""")